# R3.09 - Cryptographie et Sécurité
## TP 4 : Chiffrement par bloc
**Enseignant :** B. Colombel  
**Étudiant :** Mamadou-Bailo BARRY

---
## 1. AES avec OpenSSL (Exercice 1)
Dans cet exercice, nous chiffrons l'image `tux.ppm` en ligne de commande avec OpenSSL.

### Commandes exécutées dans le terminal :
```bash
# 1. Extraire les 3 premières lignes de l'en-tête PPM (format P6, dimensions, valeur max 255)
head -n 3 tux.ppm > header.txt

# 2. Extraire tout le reste du fichier (les pixels) dans body.bin
tail -n +4 tux.ppm > body.bin

# 3. Chiffrer body.bin avec AES-128 en modes ECB, CBC et CTR
openssl enc -aes-128-ecb -in body.bin -out tux-aes-128-ecb.enc -pass pass:secret -pbkdf2
openssl enc -aes-128-cbc -in body.bin -out tux-aes-128-cbc.enc -pass pass:secret -pbkdf2
openssl enc -aes-128-ctr -in body.bin -out tux-aes-128-ctr.enc -pass pass:secret -pbkdf2

# 4. Remettre l'en-tête au début des fichiers chiffrés pour obtenir des images PPM visualisables
cat header.txt tux-aes-128-ecb.enc > tux-aes-128-ecb.ppm
cat header.txt tux-aes-128-cbc.enc > tux-aes-128-cbc.ppm
cat header.txt tux-aes-128-ctr.enc > tux-aes-128-ctr.ppm
```

### Observation des résultats :
- **Mode ECB (`tux-aes-128-ecb.ppm`) :** La silhouette du manchot Tux est **clairement visible** ! En effet, en mode ECB, deux blocs de texte clair identiques sont chiffrés en deux blocs de chiffré strictement identiques. Les larges zones unies (le fond blanc, le corps noir, le ventre blanc, le bec orange) conservent ainsi des motifs répétitifs visibles dans le chiffré.
- **Modes CBC et CTR (`tux-aes-128-cbc.ppm` et `tux-aes-128-ctr.ppm`) :** L'image ressemble à un bruit aléatoire parfait (neige de pixels). Le manchot a complètement disparu car le chiffrement dépend des blocs précédents (CBC) ou d'un flux pseudo-aléatoire (CTR).

---
## 2. Primitives cryptographiques et Padding (Exercice 2)

In [1]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Random.random import randrange
from Crypto.Util.strxor import strxor as xor
from Crypto.Util.number import bytes_to_long, long_to_bytes
from hashlib import md5, sha3_256, sha1

def Enc(bloc, clef):
    if not isinstance(bloc, bytes) or not isinstance(clef, bytes):
        raise TypeError("le bloc et la clef doivent être de type bytes")
    if len(bloc) != 16 or len(clef) != 16:
        raise ValueError("le bloc et la clef doivent faire 16 octets")
    return AES.new(clef, AES.MODE_ECB).encrypt(bloc)

def Dec(bloc, clef):
    if not isinstance(bloc, bytes) or not isinstance(clef, bytes):
        raise TypeError("le bloc et la clef doivent être de type bytes")
    if len(bloc) != 16 or len(clef) != 16:
        raise ValueError("le bloc et la clef doivent faire 16 octets")
    return AES.new(clef, AES.MODE_ECB).decrypt(bloc)

In [2]:
def pad(s):
    """
    Ajoute un padding bit-padding (ISO/IEC 9797-1) :
    On ajoute un bit '1' (octet 0x80) puis des bits '0' (octets 0x00)
    pour que la longueur totale soit un multiple de 16 octets (128 bits).
    """
    if not isinstance(s, bytes):
        raise TypeError("s doit être de type bytes")
    nb_octets = 16 - (len(s) % 16)
    return s + b'\x80' + b'\x00' * (nb_octets - 1)

def unpad(s):
    """
    Retire le padding. Lève ValueError si le padding est invalide.
    """
    if not isinstance(s, bytes) or len(s) == 0 or len(s) % 16 != 0:
        raise ValueError("message non correctement paddé")
    
    i = len(s) - 1
    while i >= 0 and s[i] == 0:
        i -= 1
        
    if i < 0 or s[i] != 0x80 or (len(s) - i) > 16:
        raise ValueError("message non correctement paddé")
        
    return s[:i]

In [3]:
# Tests du padding
s1 = b"hello"
p1 = pad(s1)
print(f"pad(b'hello') (taille {len(p1)}) :", p1)
print("unpad(pad(b'hello')) :", unpad(p1))

s2 = b"1234567890123456"  # exactement 16 octets
p2 = pad(s2)
print(f"pad(message de 16 octets) (taille {len(p2)}) :", p2)
assert unpad(p2) == s2

try:
    unpad(b"mauvais padding!")
except ValueError as e:
    print("Exception bien levée sur un message mal paddé :", e)

pad(b'hello') (taille 16) : b'hello\x80\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
unpad(pad(b'hello')) : b'hello'
pad(message de 16 octets) (taille 32) : b'1234567890123456\x80\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Exception bien levée sur un message mal paddé : message non correctement paddé


---
## 3. Mode ECB (Exercice 3)

In [4]:
def Enc_ECB(clair, clef):
    """Chiffre un message clair avec AES en mode ECB."""
    clair_padde = pad(clair)
    chiffre = b""
    for i in range(0, len(clair_padde), 16):
        bloc = clair_padde[i:i+16]
        chiffre += Enc(bloc, clef)
    return chiffre

def Dec_ECB(chiffre, clef):
    """Déchiffre un message avec AES en mode ECB."""
    if len(chiffre) % 16 != 0:
        raise ValueError("la taille du chiffré doit être un multiple de 16")
    clair_padde = b""
    for i in range(0, len(chiffre), 16):
        bloc = chiffre[i:i+16]
        clair_padde += Dec(bloc, clef)
    return unpad(clair_padde)

In [5]:
clef = get_random_bytes(16)
message = b"Bonjour, ceci est un test de chiffrement en mode ECB !"
chiffre = Enc_ECB(message, clef)
dechiffre = Dec_ECB(chiffre, clef)
print("Message clair :", message)
print("Déchiffré ECB :", dechiffre)
print("Test ECB réussi :", dechiffre == message)

Message clair : b'Bonjour, ceci est un test de chiffrement en mode ECB !'
Déchiffré ECB : b'Bonjour, ceci est un test de chiffrement en mode ECB !'
Test ECB réussi : True


---
## 4. Mode CBC (Exercice 4)

In [6]:
def Enc_CBC(clair, clef):
    """Chiffre un message clair avec AES en mode CBC avec un IV aléatoire."""
    clair_padde = pad(clair)
    iv = get_random_bytes(16)
    chiffre = iv
    c_prec = iv
    for i in range(0, len(clair_padde), 16):
        bloc = clair_padde[i:i+16]
        c_actuel = Enc(xor(bloc, c_prec), clef)
        chiffre += c_actuel
        c_prec = c_actuel
    return chiffre

def Dec_CBC(chiffre, clef):
    """Déchiffre un message chiffré en mode CBC."""
    if len(chiffre) < 16 or len(chiffre) % 16 != 0:
        raise ValueError("la taille du chiffré doit être un multiple de 16 au moins égal à 16")
    iv = chiffre[:16]
    c_prec = iv
    clair_padde = b""
    for i in range(16, len(chiffre), 16):
        c_actuel = chiffre[i:i+16]
        bloc = xor(Dec(c_actuel, clef), c_prec)
        clair_padde += bloc
        c_prec = c_actuel
    return unpad(clair_padde)

In [7]:
clef = get_random_bytes(16)
message = b"Test du mode CBC avec vecteur d'initialisation aléatoire (IV)"
chiffre = Enc_CBC(message, clef)
dechiffre = Dec_CBC(chiffre, clef)
print("Message clair :", message)
print("Déchiffré CBC :", dechiffre)
print("Test CBC réussi :", dechiffre == message)

Message clair : b"Test du mode CBC avec vecteur d'initialisation aléatoire (IV)"
Déchiffré CBC : b"Test du mode CBC avec vecteur d'initialisation aléatoire (IV)"
Test CBC réussi : True


---
## 5. Attaque sur ECB et Image Mystère (Exercice 5)

### 1. Questions théoriques :
- **(a) Taille du fichier en octets (hors en-tête) pour une dimension $L \times H$ :**
  Chaque pixel au format PPM P6 est codé sur 3 octets (Rouge, Vert, Bleu). Donc la taille du corps est :
  $$\text{Taille} = L \times H \times 3 \text{ octets}$$

- **(b) Nombre approximatif de pixels pour un fichier de $N$ octets :**
  L'en-tête mesure environ 15 à 20 octets (par exemple `P6\n... ...\n255\n`).
  $$\text{Nombre de pixels} \approx \frac{N - 17}{3} \approx \frac{N}{3}$$

- **(c) Dimensions possibles $(L, H)$ :**
  On cherche deux entiers positifs tels que $L \times H \approx \frac{N}{3}$. Selon le ratio de l'image (ex: 16/9, 4/3, 3/2), si $r = L/H$, alors $H \approx \sqrt{\frac{N}{3r}}$ et $L = r \times H$.

### 2. Résolution pour `image.ecb.ppm` :
- Taille du fichier : $N = 28\,553\,329$ octets.
- Nombre de pixels estimé : $28\,553\,329 / 3 \approx 9\,517\,776$ pixels.
- En testant les dimensions et le calage des lignes, les dimensions exactes sont :
  **$L = 3979$ pixels, $H = 2392$ pixels**.
  Vérification : $3979 \times 2392 \times 3 = 28\,553\,304$ octets.
  En-tête PPM : `P6\n3979 2392\n255\n` (17 octets).
  Total = $17 + 28\,553\,304 = 28\,553\,321$ octets (+ 7 octets de padding PKCS#7 et 1 octet de retour à la ligne = $28\,553\,329$ octets).

In [8]:
L, H = 3979, 2392
taille_pixels = L * H * 3
en_tete = f"P6\n{L} {H}\n255\n".encode("ascii")

with open("image.ecb.ppm", "rb") as f:
    data = f.read()

# Les 17 premiers octets correspondent au chiffrement de l'en-tête original
image_ppm = en_tete + data[17:17+taille_pixels]

with open("velo-revele.ppm", "wb") as f:
    f.write(image_ppm)

print("Image révélée avec succès dans velo-revele.ppm !")
print(f"Dimensions : {L} x {H} pixels")
print("Contenu identifié : Un vélo de course / bicyclette")

Image révélée avec succès dans velo-revele.ppm !
Dimensions : 3979 x 2392 pixels
Contenu identifié : Un vélo de course / bicyclette
